In [315]:
import numpy as np
import matplotlib.pyplot as plt
import time as tm

In [316]:
def Adaptive_Forward_Difference_21(f,t,epsilon,rl,ru,inter):
    # algoritmo 2.1
    h=2*np.sqrt(epsilon)/np.sqrt(3)
    l=0
    u=np.infty
    i=0
    while True:
        r=np.abs(f(t+4*h)-4*f(t+h)+3*f(t))/(8*epsilon)
        if r<rl:
            l=h
        elif r>ru:
            u=h
        else:
            break
        if u==np.infty:
            h*=4
        elif l==0:
            h/=4
        else:
            h=(l+u)/2
        if i>inter:
            break
        i+=1
    return h




In [ ]:
def r_s(f,t,s,w,h,epsilon):
    numerator=np.abs(np.sum(w*np.array([f(t+h*sj) for sj in s])))
    return numerator / epsilon

def Adaptive_Forward_Difference_31(f,t,s,w,h0,eta,epsilon,rl,ru,max_iter=20):
    """
    Estimación Adaptativa del Intervalo de Diferencias Finitas (Algoritmo 3.1).

    Parámetros:
        f: Función ruidosa a derivar.
        t: Punto en el que se estima la derivada.
        s: Desplazamientos para el esquema de diferencias finitas.
        w: Pesos para el esquema de diferencias finitas.
        h0: Intervalo inicial.
        eta: Factor de escala para el ajuste del intervalo.
        epsilon: Nivel de ruido de la función.
        d: Orden de la derivada a aproximar.
        rl: Cota inferior para la razón de prueba.
        ru: Cota superior para la razón de prueba.
        max_iter: Número máximo de iteraciones (por defecto: 20).

    Devuelve:
        h: Intervalo estimado para diferencias finitas.
    """
    h=h0
    l=0
    u=np.inf
    iter_count=0
    
    while iter_count<max_iter:
        # computa el razon
        r=r_s(f,t,s,w,h,epsilon)
        
        if r<rl:
            l=h
        elif r>ru:
            u=h
        else:
            break
        
        #  actualiza h
        if u==np.inf:
            h*=eta
        elif l==0:
            h/=eta
        else:
            h=(l+u)/2
        
        iter_count+=1
    
    return h

In [318]:
def first_order_central_diff(f,t,h):
    # primera derivada central
    v_d_f=[]
    for i in range(len(t)):
        ei=np.zeros_like(t)
        ei[i]=1
        fs_r=(f(t+h*ei)-f(t-h*ei))/(2*h) 
        v_d_f.append(fs_r)   

    return np.array(v_d_f)

def second_order_central_diff(f,t,h):
    # segunda derivada central
    v_d_f=[]
    for i in range(len(t)):
        ei=np.zeros_like(t)
        ei[i]=1
        fs_r=(f(t+h*ei)-2*f(t)-f(t-h*ei))/(h**2)    
        v_d_f.append(fs_r)   

    return np.array(v_d_f)

def diff_d_th(f,t,s,w,h,d):
    # derivada d-th orden
    v_d_f=[]
    for i in range(len(t)):
        ei=np.zeros_like(t)
        ei[i]=1
        suma=0
        for sj,wj in zip(s,w):
            suma+=wj*f(t+h*sj*ei)
        v_d_f.append(suma/h**d)
    return np.array(v_d_f)


In [319]:
def backtracking(fun,x,fun_xk,p,grad_xk,alpha,c_1,rho,inter_b):
    alpha_i=np.copy(alpha)
    for i in range(inter_b):
        fun_alphai=fun(x+alpha_i*p)
        cond=fun_alphai<=fun_xk+c_1*alpha_i*np.dot(grad_xk,p)
        if cond:
            return  alpha_i, i
        alpha_i*=rho
    return  max(alpha_i,1e-8), i

def BFGS_mod(f,g,H,tao,N,x,alpha,c_1,inter_b,rho_back,lambda_1=10**(-5),lambda_2=10**(-5),guardar=False):
    v_x=[]
    v_g=[]
    v_f=[]
    x_i=x
    f_i=f(x)
    g_i=g(x_i)
    H_i=np.copy(H)
    dim=H.shape
    I=np.eye(dim[0])
    res=0
    alpha_g=np.copy(alpha)
    for i in range(N):
        alpha=np.copy(alpha_g)
        n_g_i=np.linalg.norm(g_i)
        if guardar:
            v_x.append(np.copy(x_i))
            v_g.append(np.copy(g_i))
            v_f.append(np.copy(f_i))
        if n_g_i<tao:
            res=1
            break
        p_i=-np.dot(H_i,g_i)
        pg=np.dot(p_i,g_i)
        if pg>0:
            gg=np.dot(g_i,g_i)
            lambda_1=10**(-5)+pg/gg
            H_i+=lambda_1*I
            p_i-=lambda_1*g_i
        alpha,j=backtracking(f,x_i,f_i,p_i,g_i,alpha,c_1,rho_back,inter_b)
        x_ip1=x_i+alpha*p_i
        s_i=x_ip1-x_i
        g_ip1=g(x_ip1)
        y_i=g_ip1-g_i
        ys=np.dot(y_i,s_i)
        if ys<=0: 
            lambda_2=10**(-5)-ys/np.dot(y_i,y_i)
            H_i+=lambda_2*I
        else:
            rho=1/ys
            sy=np.outer(s_i,y_i)
            sy_o=np.outer(y_i,s_i)
            H_i=np.dot(I-rho*sy,np.dot(H_i,(I-rho*sy_o)))+rho*np.outer(s_i,s_i)
        x_i=x_ip1
        g_i=g_ip1
        f_i=f(x_i)

    if guardar:
        return  x_i,g_i,f_i,i,np.array(v_x),np.array(v_g),np.array(v_f),res
    else:
        return x_i,g_i,f_i,i,res

In [320]:

def fun_op(x,alpha,beta,gamma):
    """
    Función objetivo para optimización de mezcla de combustibles
    
    Args:
        x (np.array): Vector de proporciones de aditivos (50 elementos)
        alpha (np.array): Coeficientes de términos cuadráticos (50 elementos)
        beta (np.array): Matriz 50x50 de interacciones (triangular superior)
        gamma (float): Coeficiente de penalización
    
    Returns:
        float: Valor de la función objetivo
    """
    # Términos cuadráticos
    f_1=np.sum(alpha*x**2)
    
    # Términos de interacción (usando vectorización)
    xx=np.outer(x,x)  # Producto externo x_i*x_j
    xx=np.clip(xx,-100,100)  # Límites seguros
    exp_terms=np.where(xx!=0, xx*np.exp(-xx),0.0)
    
    # Solo consideramos i < j (triangular superior sin diagonal)
    f_2=np.sum(beta*np.triu(exp_terms, k=1))
    
    # Término de penalización
    f_3=gamma*(np.sum(x)-1)**4
    
    return f_1+f_2+f_3

In [321]:
def exact_gradient_vec(x,alpha,beta,gamma):
    grad=2*alpha*x
    
    # Matriz de contribuciones exponenciales
    xx=np.outer(x,x)
    exp_terms=(1-xx)*np.exp(-xx)
    
    # Suma sobre j (considerando beta es triangular superior)
    grad+=np.sum(beta*np.triu(exp_terms,k=1),axis=1)  # Suma sobre j > i
    grad+=np.sum(beta.T*np.tril(exp_terms,k=-1),axis=0)  # Suma sobre j < i
    
    # Penalización
    grad+=4*gamma*(np.sum(x)-1)**3
    
    return grad

In [322]:
# Parámetros de ejemplo
n=50  # Número de variables
np.random.seed(42)
x_ini=np.random.rand(n)
alpha=np.random.uniform(0.1,0.5,n)
beta=np.triu(np.random.uniform(-0.2,0.2,(n,n)),k=1)
gamma=10.0



In [323]:
def op(x):
    return fun_op(x,alpha,beta,gamma)

In [324]:
epsilon=0.001
rl=1.5
ru=6
def gra_f_op_0(x):
    
    h=Adaptive_Forward_Difference_21(op,x,epsilon,rl,ru,100)
    g=first_order_central_diff(op,x,h)
    return g

In [325]:
def gra_f_op_1(x):
    g=exact_gradient_vec(x,alpha,beta,gamma)
    return g

In [326]:
hess=np.eye(n)
tm_0=tm.time()
sol_k,g_k,f_k,i,v_sol_k,_,_,res=BFGS_mod(f=op,g=gra_f_op_0,H=hess,tao=10**(-6),N=10000,x=np.copy(x_ini),
                                     alpha=1.0,c_1=0.1,inter_b=100,rho_back=0.6,guardar=True)
tm_1=tm.time()

print("time:",tm_1-tm_0)
print("f(x_ini):",op(x_ini))
print("iterracciones:",i)
print("res:",res)
print("norma de g_k",np.linalg.norm(g_k))
po=6

if len(v_sol_k)<=po:
    print(v_sol_k)
else:
    print(v_sol_k[:int(po/2)])
    print(v_sol_k[-int(po/2):])


print("f(x_k):",f_k)


time: 0.24699759483337402
f(x_ini): 2056879.892935461
iterracciones: 6
res: 1
norma de g_k 0.0
[[ 0.37454012  0.95071431  0.73199394  0.59865848  0.15601864  0.15599452
   0.05808361  0.86617615  0.60111501  0.70807258  0.02058449  0.96990985
   0.83244264  0.21233911  0.18182497  0.18340451  0.30424224  0.52475643
   0.43194502  0.29122914  0.61185289  0.13949386  0.29214465  0.36636184
   0.45606998  0.78517596  0.19967378  0.51423444  0.59241457  0.04645041
   0.60754485  0.17052412  0.06505159  0.94888554  0.96563203  0.80839735
   0.30461377  0.09767211  0.68423303  0.44015249  0.12203823  0.49517691
   0.03438852  0.9093204   0.25877998  0.66252228  0.31171108  0.52006802
   0.54671028  0.18485446]
 [-0.28448015  0.29169299  0.07297267 -0.06036251 -0.5030006  -0.50302617
  -0.60093641  0.20715543 -0.05790549  0.0490525  -0.63843505  0.31088955
   0.17342126 -0.44668128 -0.47719501 -0.47561474 -0.35477765 -0.13426433
  -0.22707494 -0.36779132 -0.04716836 -0.51952738 -0.36687563 -0

In [327]:
hess=np.eye(n)
tm_0=tm.time()
sol_k,g_k,f_k,i,v_sol_k,_,_,res=BFGS_mod(f=op,g=gra_f_op_1,H=hess,tao=10**(-6),N=10000,x=np.copy(x_ini),
                                     alpha=1.0,c_1=0.1,inter_b=100,rho_back=0.6,guardar=True)
tm_1=tm.time()

print("time:",tm_1-tm_0)
print("f(x_ini):",op(x_ini))
print("iterracciones:",i)
print("res:",res)
print("norma de g_k",np.linalg.norm(g_k))
po=6

if len(v_sol_k)<=po:
    print(v_sol_k)
else:
    print(v_sol_k[:int(po/2)])
    print(v_sol_k[-int(po/2):])


print("f(x_k):",f_k)


C:\Users\chris\AppData\Local\Temp\ipykernel_22832\3711480531.py:6: RuntimeWarning: overflow encountered in exp
  exp_terms=(1-xx)*np.exp(-xx)
c:\Users\chris\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\core\fromnumeric.py:88: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


time: 175.87505793571472
f(x_ini): 2056879.892935461
iterracciones: 9999
res: 0
norma de g_k nan
[[ 0.37454012  0.95071431  0.73199394  0.59865848  0.15601864  0.15599452
   0.05808361  0.86617615  0.60111501  0.70807258  0.02058449  0.96990985
   0.83244264  0.21233911  0.18182497  0.18340451  0.30424224  0.52475643
   0.43194502  0.29122914  0.61185289  0.13949386  0.29214465  0.36636184
   0.45606998  0.78517596  0.19967378  0.51423444  0.59241457  0.04645041
   0.60754485  0.17052412  0.06505159  0.94888554  0.96563203  0.80839735
   0.30461377  0.09767211  0.68423303  0.44015249  0.12203823  0.49517691
   0.03438852  0.9093204   0.25877998  0.66252228  0.31171108  0.52006802
   0.54671028  0.18485446]
 [-0.28448029  0.29169295  0.0729722  -0.06036399 -0.50299944 -0.50302694
  -0.60093617  0.20715538 -0.05790776  0.04905415 -0.63843346  0.31089208
   0.17342082 -0.44668244 -0.47719262 -0.47561218 -0.35477831 -0.13426434
  -0.22707527 -0.36779035 -0.04716956 -0.51952745 -0.36687716 

In [ ]:


def fun_op_softmax(x,alpha,beta,gamma):
    """
    Versión estable con softmax para las interacciones
    
    Args:
        x (np.array): Vector de proporciones (50 elementos)
        alpha (np.array): Coeficientes cuadráticos (50,)
        beta (np.array): Matriz triangular superior (50x50)
        gamma (float): Peso de penalización
    
    Returns:
        float: Valor de la función objetivo
    """
    # 1. Término cuadrático (estable)
    f_1=np.sum(alpha * x**2)
    
    # 2. Interacciones suavizadas (softmax)
    xx=np.outer(x, x)
    interaction_terms=xx/(1+np.abs(xx))  # Evita divisiones por cero
    f_2=np.sum(beta*np.triu(interaction_terms,k=1))
    
    # 3. Penalización (suavizada)
    sum_x=np.sum(x)
    f_3=gamma*(sum_x-1)**4
    
    return f_1+f_2+f_3

In [329]:
def grad_softmax(x,alpha,beta,gamma):
    """
    Gradiente exacto de fun_op_softmax
    
    Args:
        x (np.array): Vector de variables (50,)
        alpha, beta, gamma: Parámetros como en fun_op_softmax
    
    Returns:
        np.array: Gradiente (50,)
    """
    # 1. Derivada términos cuadráticos
    grad=2*alpha*x
    
    # 2. Derivada de interacciones softmax
    for k in range(len(x)):
        for j in range(len(x)):
            if k!=j:
                xk=x[k]
                xj=x[j]
                denom=(1+np.abs(xk*xj))**2
                grad[k]+=beta[min(k,j),max(k,j)]*xj/denom
    
    # 3. Derivada penalización
    grad+=4*gamma*(np.sum(x)-1)**3
    
    return grad

In [330]:
def prueba_0(x_ini,alpha,beta,gamma):

    # Función objetivo y gradiente
    def op(x):
        return fun_op_softmax(x,alpha,beta,gamma)

    def grad_op(x):
        return grad_softmax(x,alpha,beta,gamma)

    hess=np.eye(n)
    tm_0=tm.time()
    sol_k,g_k,f_k,i,v_sol_k,_,_,res=BFGS_mod(f=op,g=grad_op,H=hess,tao=10**(-6),N=10000,x=np.copy(x_ini),
                                        alpha=1.0,c_1=0.1,inter_b=100,rho_back=0.6,guardar=True)
    tm_1=tm.time()

    print("time:",tm_1-tm_0)
    print("f(x_ini):",op(x_ini))
    print("iterracciones:",i)
    print("res:",res)
    print("norma de g_k",np.linalg.norm(g_k))
    po=6

    if len(v_sol_k)<=po:
        print(v_sol_k)
    else:
        print(v_sol_k[:int(po/2)])
        print(v_sol_k[-int(po/2):])


    print("f(x_k):",f_k)


In [331]:
def prueba_1(x_ini,alpha,beta,gamma):
    hess=np.eye(n)
    tm_0=tm.time()
    epsilon=0.001
    rl=1.5
    ru=6
    def op(x):
        return fun_op_softmax(x,alpha,beta,gamma)
    def gra_f_op_0(x):   
        h=Adaptive_Forward_Difference_21(op,x,epsilon,rl,ru,100)
        g=first_order_central_diff(op,x,h)
        return g
    sol_k,g_k,f_k,i,v_sol_k,_,_,res=BFGS_mod(f=op,g=gra_f_op_0,H=hess,tao=10**(-6),N=10000,x=np.copy(x_ini),
                                        alpha=1.0,c_1=0.1,inter_b=100,rho_back=0.6,guardar=True)
    tm_1=tm.time()

    print("time:",tm_1-tm_0)
    print("f(x_ini):",op(x_ini))
    print("iterracciones:",i)
    print("res:",res)
    print("norma de g_k",np.linalg.norm(g_k))
    po=6

    if len(v_sol_k)<=po:
        print(v_sol_k)
    else:
        print(v_sol_k[:int(po/2)])
        print(v_sol_k[-int(po/2):])


    print("f(x_k):",f_k)

In [332]:
def prueba_2(x,alpha,beta,gamma):
    hess=np.eye(n)
    tm_0=tm.time()
    epsilon=0.001
    rl=1.5
    ru=6
    def op(x):
        return fun_op_softmax(x,alpha,beta,gamma)
    def gra_f_op_0(x):   
        s=np.array([1,-1])
        w=np.array([1/2,1/2])
        h=Adaptive_Forward_Difference_31(op,x,s,w,0.01,2,epsilon,rl,ru,100)
        g=first_order_central_diff(op,x,h)
        return g
    sol_k,g_k,f_k,i,v_sol_k,_,_,res=BFGS_mod(f=op,g=gra_f_op_0,H=hess,tao=10**(-6),N=10000,x=np.copy(x_ini),
                                        alpha=1.0,c_1=0.1,inter_b=100,rho_back=0.6,guardar=True)
    tm_1=tm.time()

    print("time:",tm_1-tm_0)
    print("f(x_ini):",op(x_ini))
    print("iterracciones:",i)
    print("res:",res)
    print("norma de g_k",np.linalg.norm(g_k))
    po=6

    if len(v_sol_k)<=po:
        print(v_sol_k)
    else:
        print(v_sol_k[:int(po/2)])
        print(v_sol_k[-int(po/2):])


    print("f(x_k):",f_k)

In [333]:
def prueba_3(x_ini,alpha,beta,gamma):
    hess=np.eye(n)
    tm_0=tm.time()
    epsilon=0.001
    rl=1.5
    ru=6
    def op(x):
        return fun_op_softmax(x,alpha,beta,gamma)
    def gra_f_op_0(x):   
        h=0.0001
        g=first_order_central_diff(op,x,h)
        return g
    sol_k,g_k,f_k,i,v_sol_k,_,_,res=BFGS_mod(f=op,g=gra_f_op_0,H=hess,tao=10**(-6),N=10000,x=np.copy(x_ini),
                                        alpha=1.0,c_1=0.1,inter_b=100,rho_back=0.6,guardar=True)
    tm_1=tm.time()

    print("time:",tm_1-tm_0)
    print("f(x_ini):",op(x_ini))
    print("iterracciones:",i)
    print("res:",res)
    print("norma de g_k",np.linalg.norm(g_k))
    po=6

    if len(v_sol_k)<=po:
        print(v_sol_k)
    else:
        print(v_sol_k[:int(po/2)])
        print(v_sol_k[-int(po/2):])


    print("f(x_k):",f_k)

In [334]:
# Parámetros
n=50
np.random.seed(42)
x_ini=np.random.rand(n)
x_ini=x_ini / np.sum(x_ini)  # Normalizado

alpha = np.random.uniform(0.1, 0.5, n)
beta = np.triu(np.random.uniform(-0.2, 0.2, (n, n)), k=1)
gamma = 10.0

In [335]:
prueba_0(x_ini,alpha,beta,gamma)

time: 0.2838761806488037
f(x_ini): 0.008905732816352624
iterracciones: 44
res: 1
norma de g_k 4.2979646962120147e-07
[[ 0.01679839  0.0426402   0.03283044  0.02685025  0.00699755  0.00699646
   0.00260509  0.03884861  0.02696043  0.03175755  0.00092323  0.04350114
   0.03733564  0.00952356  0.00815498  0.00822582  0.01364548  0.02353569
   0.01937304  0.01306183  0.02744203  0.0062564   0.01310289  0.01643159
   0.02045506  0.03521569  0.00895551  0.02306378  0.02657021  0.00208333
   0.02724881  0.00764813  0.00291761  0.04255818  0.04330927  0.03625719
   0.01366214  0.00438066  0.03068833  0.01974115  0.0054735   0.02220903
   0.00154235  0.04078366  0.01160646  0.02971459  0.01398046  0.02332542
   0.02452034  0.00829085]
 [ 0.01643416  0.03243629  0.02403848  0.02096192  0.01278738  0.00187326
   0.00218875  0.02915829  0.0224587   0.03539567  0.00328136  0.03863085
   0.02497929  0.00714492  0.00811984  0.01373239  0.01551205  0.01960602
   0.02233378  0.00855558  0.01853383 -0.0

In [336]:
prueba_1(x_ini,alpha,beta,gamma)

time: 0.35987043380737305
f(x_ini): 0.008905732816352624
iterracciones: 48
res: 1
norma de g_k 7.071606294266743e-07
[[ 0.01679839  0.0426402   0.03283044  0.02685025  0.00699755  0.00699646
   0.00260509  0.03884861  0.02696043  0.03175755  0.00092323  0.04350114
   0.03733564  0.00952356  0.00815498  0.00822582  0.01364548  0.02353569
   0.01937304  0.01306183  0.02744203  0.0062564   0.01310289  0.01643159
   0.02045506  0.03521569  0.00895551  0.02306378  0.02657021  0.00208333
   0.02724881  0.00764813  0.00291761  0.04255818  0.04330927  0.03625719
   0.01366214  0.00438066  0.03068833  0.01974115  0.0054735   0.02220903
   0.00154235  0.04078366  0.01160646  0.02971459  0.01398046  0.02332542
   0.02452034  0.00829085]
 [ 0.01643416  0.03243629  0.02403848  0.02096192  0.01278738  0.00187326
   0.00218875  0.02915829  0.0224587   0.03539567  0.00328135  0.03863085
   0.02497929  0.00714492  0.00811984  0.01373239  0.01551205  0.01960602
   0.02233378  0.00855558  0.01853383 -0.0

In [337]:
prueba_2(x_ini,alpha,beta,gamma)

time: 0.028183460235595703
f(x_ini): 0.008905732816352624
iterracciones: 0
res: 1
norma de g_k 0.0
[[0.01679839 0.0426402  0.03283044 0.02685025 0.00699755 0.00699646
  0.00260509 0.03884861 0.02696043 0.03175755 0.00092323 0.04350114
  0.03733564 0.00952356 0.00815498 0.00822582 0.01364548 0.02353569
  0.01937304 0.01306183 0.02744203 0.0062564  0.01310289 0.01643159
  0.02045506 0.03521569 0.00895551 0.02306378 0.02657021 0.00208333
  0.02724881 0.00764813 0.00291761 0.04255818 0.04330927 0.03625719
  0.01366214 0.00438066 0.03068833 0.01974115 0.0054735  0.02220903
  0.00154235 0.04078366 0.01160646 0.02971459 0.01398046 0.02332542
  0.02452034 0.00829085]]
f(x_k): 0.008905732816352624


In [338]:
prueba_3(x_ini,alpha,beta,gamma)

time: 0.26467418670654297
f(x_ini): 0.008905732816352624
iterracciones: 44
res: 1
norma de g_k 4.3218642812429644e-07
[[ 0.01679839  0.0426402   0.03283044  0.02685025  0.00699755  0.00699646
   0.00260509  0.03884861  0.02696043  0.03175755  0.00092323  0.04350114
   0.03733564  0.00952356  0.00815498  0.00822582  0.01364548  0.02353569
   0.01937304  0.01306183  0.02744203  0.0062564   0.01310289  0.01643159
   0.02045506  0.03521569  0.00895551  0.02306378  0.02657021  0.00208333
   0.02724881  0.00764813  0.00291761  0.04255818  0.04330927  0.03625719
   0.01366214  0.00438066  0.03068833  0.01974115  0.0054735   0.02220903
   0.00154235  0.04078366  0.01160646  0.02971459  0.01398046  0.02332542
   0.02452034  0.00829085]
 [ 0.01643416  0.03243629  0.02403848  0.02096192  0.01278738  0.00187326
   0.00218875  0.02915829  0.0224587   0.03539567  0.00328136  0.03863085
   0.02497929  0.00714492  0.00811984  0.01373239  0.01551205  0.01960602
   0.02233378  0.00855558  0.01853383 -0.

In [339]:
# Parámetros
n=10
np.random.seed(42)
x_ini=np.random.rand(n)
x_ini=x_ini / np.sum(x_ini)  # Normalizado

alpha = np.random.uniform(0.1, 0.5, n)
beta = np.triu(np.random.uniform(-0.2, 0.2, (n, n)), k=1)
gamma = 10.0

In [340]:
prueba_0(x_ini,alpha,beta,gamma)

time: 0.021843433380126953
f(x_ini): 0.02207604755325864
iterracciones: 28
res: 1
norma de g_k 2.0642369500407235e-07
[[ 0.07200801  0.18278161  0.14073106  0.11509637  0.0299957   0.02999106
   0.01116699  0.16652855  0.11556865  0.13613201]
 [ 0.0991032   0.09100598  0.08727522  0.08644847 -0.00563891  0.02061542
   0.02493927  0.15385326  0.11379663  0.15157424]
 [ 0.17598843 -0.00280536  0.04496433  0.08414809 -0.02157062  0.0455571
   0.08019475  0.1696256   0.14676512  0.21159974]]
[[ 0.05755482  0.03009276 -0.02197677  0.10864786 -0.2666329   0.29780154
  -0.17726434  0.30320586  0.2299517   0.388863  ]
 [ 0.05758213  0.03010804 -0.02196043  0.10862845 -0.26663469  0.29778196
  -0.17725627  0.30319833  0.22994027  0.38887416]
 [ 0.05758426  0.03010855 -0.02195977  0.10862785 -0.26663519  0.2977819
  -0.17725572  0.30319742  0.22993894  0.38887514]]
f(x_k): -0.007363088087755326


In [341]:
prueba_1(x_ini,alpha,beta,gamma)

C:\Users\chris\AppData\Local\Temp\ipykernel_22832\2051219467.py:47: RuntimeWarning: invalid value encountered in scalar divide
  lambda_2=10**(-5)-ys/np.dot(y_i,y_i)


time: 76.33936667442322
f(x_ini): 0.02207604755325864
iterracciones: 9999
res: 0
norma de g_k nan
[[ 0.07200801  0.18278161  0.14073106  0.11509637  0.0299957   0.02999106
   0.01116699  0.16652855  0.11556865  0.13613201]
 [ 0.09910322  0.09100599  0.08727524  0.08644846 -0.00563894  0.02061542
   0.02493928  0.15385329  0.11379664  0.15157427]
 [ 0.17605775 -0.00293693  0.0449044   0.08412145 -0.0216008   0.04556389
   0.08024638  0.16963554  0.14679045  0.21164961]]
[[nan nan nan nan nan nan nan nan nan nan]
 [nan nan nan nan nan nan nan nan nan nan]
 [nan nan nan nan nan nan nan nan nan nan]]
f(x_k): nan


In [342]:
prueba_2(x_ini,alpha,beta,gamma)

time: 0.03316307067871094
f(x_ini): 0.02207604755325864
iterracciones: 0
res: 1
norma de g_k 0.0
[[0.07200801 0.18278161 0.14073106 0.11509637 0.0299957  0.02999106
  0.01116699 0.16652855 0.11556865 0.13613201]]
f(x_k): 0.02207604755325864


In [343]:
prueba_3(x_ini,alpha,beta,gamma)

time: 0.1403801441192627
f(x_ini): 0.02207604755325864
iterracciones: 28
res: 1
norma de g_k 2.0639658442271141e-07
[[ 0.07200801  0.18278161  0.14073106  0.11509637  0.0299957   0.02999106
   0.01116699  0.16652855  0.11556865  0.13613201]
 [ 0.0991032   0.09100598  0.08727522  0.08644847 -0.00563891  0.02061542
   0.02493927  0.15385326  0.11379663  0.15157424]
 [ 0.17598844 -0.00280537  0.04496432  0.08414809 -0.02157063  0.04555711
   0.08019476  0.1696256   0.14676512  0.21159974]]
[[ 0.05755485  0.03009277 -0.02197675  0.10864786 -0.2666329   0.29780151
  -0.17726432  0.30320586  0.2299517   0.38886301]
 [ 0.05758216  0.03010805 -0.02196042  0.10862846 -0.26663469  0.29778194
  -0.17725625  0.30319833  0.22994027  0.38887417]
 [ 0.05758429  0.03010856 -0.02195977  0.10862786 -0.26663519  0.29778189
  -0.17725571  0.30319742  0.22993893  0.38887515]]
f(x_k): -0.007363088087755011


In [344]:
# Parámetros
n=25
np.random.seed(42)
x_ini=np.random.rand(n)
x_ini=x_ini / np.sum(x_ini)  # Normalizado

alpha = np.random.uniform(0.1, 0.5, n)
beta = np.triu(np.random.uniform(-0.2, 0.2, (n, n)), k=1)
gamma = 10.0

In [345]:
prueba_0(x_ini,alpha,beta,gamma)

time: 0.3537306785583496
f(x_ini): 0.01272981425700074
iterracciones: 53
res: 1
norma de g_k 6.769194999110421e-07
[[ 3.39874022e-02  8.62719583e-02  6.64243195e-02  5.43248792e-02
   1.41578112e-02  1.41556224e-02  5.27076003e-03  7.86005973e-02
   5.45477952e-02  6.42535907e-02  1.86792670e-03  8.80138458e-02
   7.55394721e-02  1.92685760e-02  1.64995897e-02  1.66429243e-02
   2.76082667e-02  4.76186849e-02  3.91965729e-02  2.64274010e-02
   5.55221975e-02  1.26582807e-02  2.65104783e-02  3.32452699e-02
   4.13857774e-02]
 [ 2.47788036e-02  8.39077251e-02  5.95143058e-02  5.07321699e-02
   1.32192660e-02  1.56149119e-05  7.85645703e-03  7.49957724e-02
   4.59378480e-02  6.05369884e-02  1.18924938e-02  8.06112515e-02
   7.15942646e-02  7.65756167e-03  1.35757308e-02  1.03742855e-02
   1.82944402e-02  3.77909252e-02  3.46271245e-02  3.06545932e-02
   4.18501476e-02  9.28251373e-03  3.00799611e-02  2.50350564e-02
   3.98487507e-02]
 [-6.34165281e-03  1.08113470e-01  4.67304122e-02  6.24

In [346]:
prueba_1(x_ini,alpha,beta,gamma)

C:\Users\chris\AppData\Local\Temp\ipykernel_22832\2051219467.py:47: RuntimeWarning: invalid value encountered in scalar divide
  lambda_2=10**(-5)-ys/np.dot(y_i,y_i)


time: 174.25476360321045
f(x_ini): 0.01272981425700074
iterracciones: 9999
res: 0
norma de g_k nan
[[ 3.39874022e-02  8.62719583e-02  6.64243195e-02  5.43248792e-02
   1.41578112e-02  1.41556224e-02  5.27076003e-03  7.86005973e-02
   5.45477952e-02  6.42535907e-02  1.86792670e-03  8.80138458e-02
   7.55394721e-02  1.92685760e-02  1.64995897e-02  1.66429243e-02
   2.76082667e-02  4.76186849e-02  3.91965729e-02  2.64274010e-02
   5.55221975e-02  1.26582807e-02  2.65104783e-02  3.32452699e-02
   4.13857774e-02]
 [ 2.47788035e-02  8.39077251e-02  5.95143060e-02  5.07321700e-02
   1.32192659e-02  1.56146093e-05  7.85645697e-03  7.49957725e-02
   4.59378481e-02  6.05369886e-02  1.18924331e-02  8.06112514e-02
   7.15942645e-02  7.65756147e-03  1.35757308e-02  1.03742855e-02
   1.82944400e-02  3.77909250e-02  3.46271246e-02  3.06545934e-02
   4.18501475e-02  9.28251368e-03  3.00799613e-02  2.50350564e-02
   3.98487508e-02]
 [-6.34700884e-03  1.08115882e-01  4.67278193e-02  6.24798315e-02
   4.

In [347]:
prueba_2(x_ini,alpha,beta,gamma)

time: 0.052828311920166016
f(x_ini): 0.01272981425700074
iterracciones: 0
res: 1
norma de g_k 0.0
[[0.0339874  0.08627196 0.06642432 0.05432488 0.01415781 0.01415562
  0.00527076 0.0786006  0.0545478  0.06425359 0.00186793 0.08801385
  0.07553947 0.01926858 0.01649959 0.01664292 0.02760827 0.04761868
  0.03919657 0.0264274  0.0555222  0.01265828 0.02651048 0.03324527
  0.04138578]]
f(x_k): 0.01272981425700074


In [348]:
prueba_3(x_ini,alpha,beta,gamma)

time: 0.5446736812591553
f(x_ini): 0.01272981425700074
iterracciones: 54
res: 1
norma de g_k 9.347675309377114e-08
[[ 3.39874022e-02  8.62719583e-02  6.64243195e-02  5.43248792e-02
   1.41578112e-02  1.41556224e-02  5.27076003e-03  7.86005973e-02
   5.45477952e-02  6.42535907e-02  1.86792670e-03  8.80138458e-02
   7.55394721e-02  1.92685760e-02  1.64995897e-02  1.66429243e-02
   2.76082667e-02  4.76186849e-02  3.91965729e-02  2.64274010e-02
   5.55221975e-02  1.26582807e-02  2.65104783e-02  3.32452699e-02
   4.13857774e-02]
 [ 2.47788036e-02  8.39077251e-02  5.95143058e-02  5.07321699e-02
   1.32192660e-02  1.56149113e-05  7.85645703e-03  7.49957724e-02
   4.59378480e-02  6.05369884e-02  1.18924938e-02  8.06112515e-02
   7.15942646e-02  7.65756166e-03  1.35757308e-02  1.03742855e-02
   1.82944402e-02  3.77909252e-02  3.46271245e-02  3.06545932e-02
   4.18501476e-02  9.28251373e-03  3.00799611e-02  2.50350564e-02
   3.98487507e-02]
 [-6.34163942e-03  1.08113466e-01  4.67304201e-02  6.24

In [349]:
# Parámetros
n=100
np.random.seed(42)
x_ini=np.random.rand(n)
x_ini=x_ini / np.sum(x_ini)  # Normalizado

alpha = np.random.uniform(0.1, 0.5, n)
beta = np.triu(np.random.uniform(-0.2, 0.2, (n, n)), k=1)
gamma = 10.0

In [350]:
prueba_0(x_ini,alpha,beta,gamma)

time: 4.303084850311279
f(x_ini): 0.004351445595353904
iterracciones: 47
res: 1
norma de g_k 8.799865374422533e-07
[[ 7.96587534e-03  2.02201881e-02  1.55683522e-02  1.27325181e-02
   3.31826947e-03  3.31775647e-03  1.23534647e-03  1.84221953e-02
   1.27847646e-02  1.50595827e-02  4.37799603e-04  2.06284469e-02
   1.77047370e-02  4.51611670e-03  3.86712918e-03  3.90072355e-03
   6.47075082e-03  1.11607385e-02  9.18678667e-03  6.19398272e-03
   1.30131424e-02  2.96681356e-03  6.21345414e-03  7.79193637e-03
   9.69988649e-03  1.66994496e-02  4.24674521e-03  1.09369523e-02
   1.25997199e-02  9.87926736e-04  1.29215171e-02  3.62677813e-03
   1.38354439e-03  2.01812931e-02  2.05374645e-02  1.71933317e-02
   6.47865259e-03  2.07733123e-03  1.45525532e-02  9.36134667e-03
   2.59556004e-03  1.05316289e-02  7.31389399e-04  1.93398053e-02
   5.50384050e-03  1.40908001e-02  6.62960107e-03  1.10610234e-02
   1.16276621e-02  3.93156160e-03  2.06215299e-02  1.64858479e-02
   1.99816550e-02  1.903156

In [351]:
prueba_1(x_ini,alpha,beta,gamma)

time: 5.19846248626709
f(x_ini): 0.004351445595353904
iterracciones: 48
res: 1
norma de g_k 2.4232046976921743e-07
[[ 7.96587534e-03  2.02201881e-02  1.55683522e-02  1.27325181e-02
   3.31826947e-03  3.31775647e-03  1.23534647e-03  1.84221953e-02
   1.27847646e-02  1.50595827e-02  4.37799603e-04  2.06284469e-02
   1.77047370e-02  4.51611670e-03  3.86712918e-03  3.90072355e-03
   6.47075082e-03  1.11607385e-02  9.18678667e-03  6.19398272e-03
   1.30131424e-02  2.96681356e-03  6.21345414e-03  7.79193637e-03
   9.69988649e-03  1.66994496e-02  4.24674521e-03  1.09369523e-02
   1.25997199e-02  9.87926736e-04  1.29215171e-02  3.62677813e-03
   1.38354439e-03  2.01812931e-02  2.05374645e-02  1.71933317e-02
   6.47865259e-03  2.07733123e-03  1.45525532e-02  9.36134667e-03
   2.59556004e-03  1.05316289e-02  7.31389399e-04  1.93398053e-02
   5.50384050e-03  1.40908001e-02  6.62960107e-03  1.10610234e-02
   1.16276621e-02  3.93156160e-03  2.06215299e-02  1.64858479e-02
   1.99816550e-02  1.903156

In [352]:
prueba_2(x_ini,alpha,beta,gamma)

time: 0.589686393737793
f(x_ini): 0.004351445595353904
iterracciones: 4
res: 1
norma de g_k 0.0
[[ 7.96587534e-03  2.02201881e-02  1.55683522e-02  1.27325181e-02
   3.31826947e-03  3.31775647e-03  1.23534647e-03  1.84221953e-02
   1.27847646e-02  1.50595827e-02  4.37799603e-04  2.06284469e-02
   1.77047370e-02  4.51611670e-03  3.86712918e-03  3.90072355e-03
   6.47075082e-03  1.11607385e-02  9.18678667e-03  6.19398272e-03
   1.30131424e-02  2.96681356e-03  6.21345414e-03  7.79193637e-03
   9.69988649e-03  1.66994496e-02  4.24674521e-03  1.09369523e-02
   1.25997199e-02  9.87926736e-04  1.29215171e-02  3.62677813e-03
   1.38354439e-03  2.01812931e-02  2.05374645e-02  1.71933317e-02
   6.47865259e-03  2.07733123e-03  1.45525532e-02  9.36134667e-03
   2.59556004e-03  1.05316289e-02  7.31389399e-04  1.93398053e-02
   5.50384050e-03  1.40908001e-02  6.62960107e-03  1.10610234e-02
   1.16276621e-02  3.93156160e-03  2.06215299e-02  1.64858479e-02
   1.99816550e-02  1.90315610e-02  1.27163859e

In [353]:
prueba_3(x_ini,alpha,beta,gamma)

time: 3.346224784851074
f(x_ini): 0.004351445595353904
iterracciones: 47
res: 1
norma de g_k 8.801304096551997e-07
[[ 7.96587534e-03  2.02201881e-02  1.55683522e-02  1.27325181e-02
   3.31826947e-03  3.31775647e-03  1.23534647e-03  1.84221953e-02
   1.27847646e-02  1.50595827e-02  4.37799603e-04  2.06284469e-02
   1.77047370e-02  4.51611670e-03  3.86712918e-03  3.90072355e-03
   6.47075082e-03  1.11607385e-02  9.18678667e-03  6.19398272e-03
   1.30131424e-02  2.96681356e-03  6.21345414e-03  7.79193637e-03
   9.69988649e-03  1.66994496e-02  4.24674521e-03  1.09369523e-02
   1.25997199e-02  9.87926736e-04  1.29215171e-02  3.62677813e-03
   1.38354439e-03  2.01812931e-02  2.05374645e-02  1.71933317e-02
   6.47865259e-03  2.07733123e-03  1.45525532e-02  9.36134667e-03
   2.59556004e-03  1.05316289e-02  7.31389399e-04  1.93398053e-02
   5.50384050e-03  1.40908001e-02  6.62960107e-03  1.10610234e-02
   1.16276621e-02  3.93156160e-03  2.06215299e-02  1.64858479e-02
   1.99816550e-02  1.903156

In [354]:
# Parámetros
n=200
np.random.seed(42)
x_ini=np.random.rand(n)
x_ini=x_ini / np.sum(x_ini)  # Normalizado

alpha = np.random.uniform(0.1, 0.5, n)
beta = np.triu(np.random.uniform(-0.2, 0.2, (n, n)), k=1)
gamma = 10.0

In [355]:
prueba_0(x_ini,alpha,beta,gamma)

time: 20.571744680404663
f(x_ini): 0.0024334327334012583
iterracciones: 62
res: 1
norma de g_k 5.231784759850726e-07
[[ 3.86916627e-03  9.82130230e-03  7.56182350e-03  6.18440883e-03
   1.61174206e-03  1.61149289e-03  6.00029588e-04  8.94798544e-03
   6.20978585e-03  7.31470515e-03  2.12646996e-04  1.00196008e-02
   8.59950331e-03  2.19355760e-03  1.87833291e-03  1.89465029e-03
   3.14295790e-03  5.42096770e-03  4.46218446e-03  3.00852675e-03
   6.32071296e-03  1.44103372e-03  3.01798436e-03  3.78468104e-03
   4.71140610e-03  8.11121745e-03  2.06271912e-03  5.31227082e-03
   6.11990640e-03  4.79853456e-04  6.27620896e-03  1.76159016e-03
   6.72011934e-04  9.80241030e-03  9.97540906e-03  8.35110472e-03
   3.14679593e-03  1.00899645e-03  7.06843197e-03  4.54697134e-03
   1.26070933e-03  5.11539807e-03  3.55248742e-04  9.39368483e-03
   2.67331249e-03  6.84415033e-03  3.22011427e-03  5.37253433e-03
   5.64776073e-03  1.90962887e-03  1.00162411e-02  8.00746736e-03
   9.70544259e-03  9.2439

In [356]:
prueba_1(x_ini,alpha,beta,gamma)

time: 29.558305978775024
f(x_ini): 0.0024334327334012583
iterracciones: 59
res: 1
norma de g_k 7.290334878468228e-07
[[ 3.86916627e-03  9.82130230e-03  7.56182350e-03  6.18440883e-03
   1.61174206e-03  1.61149289e-03  6.00029588e-04  8.94798544e-03
   6.20978585e-03  7.31470515e-03  2.12646996e-04  1.00196008e-02
   8.59950331e-03  2.19355760e-03  1.87833291e-03  1.89465029e-03
   3.14295790e-03  5.42096770e-03  4.46218446e-03  3.00852675e-03
   6.32071296e-03  1.44103372e-03  3.01798436e-03  3.78468104e-03
   4.71140610e-03  8.11121745e-03  2.06271912e-03  5.31227082e-03
   6.11990640e-03  4.79853456e-04  6.27620896e-03  1.76159016e-03
   6.72011934e-04  9.80241030e-03  9.97540906e-03  8.35110472e-03
   3.14679593e-03  1.00899645e-03  7.06843197e-03  4.54697134e-03
   1.26070933e-03  5.11539807e-03  3.55248742e-04  9.39368483e-03
   2.67331249e-03  6.84415033e-03  3.22011427e-03  5.37253433e-03
   5.64776073e-03  1.90962887e-03  1.00162411e-02  8.00746736e-03
   9.70544259e-03  9.2439

In [357]:
prueba_2(x_ini,alpha,beta,gamma)

time: 2.3423149585723877
f(x_ini): 0.0024334327334012583
iterracciones: 3
res: 1
norma de g_k 0.0
[[ 3.86916627e-03  9.82130230e-03  7.56182350e-03  6.18440883e-03
   1.61174206e-03  1.61149289e-03  6.00029588e-04  8.94798544e-03
   6.20978585e-03  7.31470515e-03  2.12646996e-04  1.00196008e-02
   8.59950331e-03  2.19355760e-03  1.87833291e-03  1.89465029e-03
   3.14295790e-03  5.42096770e-03  4.46218446e-03  3.00852675e-03
   6.32071296e-03  1.44103372e-03  3.01798436e-03  3.78468104e-03
   4.71140610e-03  8.11121745e-03  2.06271912e-03  5.31227082e-03
   6.11990640e-03  4.79853456e-04  6.27620896e-03  1.76159016e-03
   6.72011934e-04  9.80241030e-03  9.97540906e-03  8.35110472e-03
   3.14679593e-03  1.00899645e-03  7.06843197e-03  4.54697134e-03
   1.26070933e-03  5.11539807e-03  3.55248742e-04  9.39368483e-03
   2.67331249e-03  6.84415033e-03  3.22011427e-03  5.37253433e-03
   5.64776073e-03  1.90962887e-03  1.00162411e-02  8.00746736e-03
   9.70544259e-03  9.24396515e-03  6.1765731

In [358]:
prueba_3(x_ini,alpha,beta,gamma)

time: 7.0974321365356445
f(x_ini): 0.0024334327334012583
iterracciones: 60
res: 1
norma de g_k 6.167452702901861e-07
[[ 3.86916627e-03  9.82130230e-03  7.56182350e-03  6.18440883e-03
   1.61174206e-03  1.61149289e-03  6.00029588e-04  8.94798544e-03
   6.20978585e-03  7.31470515e-03  2.12646996e-04  1.00196008e-02
   8.59950331e-03  2.19355760e-03  1.87833291e-03  1.89465029e-03
   3.14295790e-03  5.42096770e-03  4.46218446e-03  3.00852675e-03
   6.32071296e-03  1.44103372e-03  3.01798436e-03  3.78468104e-03
   4.71140610e-03  8.11121745e-03  2.06271912e-03  5.31227082e-03
   6.11990640e-03  4.79853456e-04  6.27620896e-03  1.76159016e-03
   6.72011934e-04  9.80241030e-03  9.97540906e-03  8.35110472e-03
   3.14679593e-03  1.00899645e-03  7.06843197e-03  4.54697134e-03
   1.26070933e-03  5.11539807e-03  3.55248742e-04  9.39368483e-03
   2.67331249e-03  6.84415033e-03  3.22011427e-03  5.37253433e-03
   5.64776073e-03  1.90962887e-03  1.00162411e-02  8.00746736e-03
   9.70544259e-03  9.2439

In [359]:
# Parámetros
n=400
np.random.seed(42)
x_ini=np.random.rand(n)
x_ini=x_ini / np.sum(x_ini)  # Normalizado

alpha = np.random.uniform(0.1, 0.5, n)
beta = np.triu(np.random.uniform(-0.2, 0.2, (n, n)), k=1)
gamma = 10.0

In [360]:
prueba_0(x_ini,alpha,beta,gamma)

time: 26.594881534576416
f(x_ini): 0.0008497147415212203
iterracciones: 70
res: 1
norma de g_k 9.30676394189885e-07
[[ 0.00189471  0.00480945  0.00370299 ...  0.00217018  0.00379849
   0.00381706]
 [ 0.00091882  0.00437444  0.00289647 ...  0.00238575  0.00302755
   0.0016418 ]
 [-0.00379008  0.00362576 -0.00054581 ...  0.00602174 -0.00013645
  -0.01103953]]
[[-1.89772939  0.9458549   0.88304472 ... -0.71309419  0.98068553
  -0.91069949]
 [-1.89772943  0.94585487  0.88304451 ... -0.71309431  0.98068559
  -0.91069936]
 [-1.89772944  0.94585483  0.88304453 ... -0.7130943   0.98068558
  -0.91069942]]
f(x_k): -228.00938255819392


In [361]:
prueba_1(x_ini,alpha,beta,gamma)

time: 469.9584593772888
f(x_ini): 0.0008497147415212203
iterracciones: 67
res: 1
norma de g_k 3.2861362957793234e-07
[[ 0.00189471  0.00480945  0.00370299 ...  0.00217018  0.00379849
   0.00381706]
 [ 0.00091882  0.00437444  0.00289647 ...  0.00238575  0.00302755
   0.0016418 ]
 [-0.00379009  0.00362576 -0.00054582 ...  0.00602175 -0.00013645
  -0.01103954]]
[[-2.01425062  0.89171597  0.87148317 ... -0.91281305  0.94890072
  -0.78255799]
 [-2.01425059  0.89171595  0.87148321 ... -0.91281305  0.94890073
  -0.78255796]
 [-2.0142506   0.89171592  0.87148322 ... -0.91281306  0.94890072
  -0.78255797]]
f(x_k): -226.777741394781


In [362]:
prueba_2(x_ini,alpha,beta,gamma)

time: 44.60039782524109
f(x_ini): 0.0008497147415212203
iterracciones: 3
res: 1
norma de g_k 0.0
[[ 0.00189471  0.00480945  0.00370299 ...  0.00217018  0.00379849
   0.00381706]
 [ 0.00091882  0.00437444  0.00289647 ...  0.00238575  0.00302755
   0.0016418 ]
 [-0.00379009  0.00362576 -0.00054582 ...  0.00602174 -0.00013645
  -0.01103954]
 [-0.043146    0.00546511 -0.02323085 ...  0.00613108 -0.01693363
  -0.03892738]]
f(x_k): -0.5176084158377583


In [363]:
prueba_3(x_ini,alpha,beta,gamma)

time: 469.50647497177124
f(x_ini): 0.0008497147415212203
iterracciones: 59
res: 1
norma de g_k 6.803783405909062e-07
[[ 0.00189471  0.00480945  0.00370299 ...  0.00217018  0.00379849
   0.00381706]
 [ 0.00091882  0.00437444  0.00289647 ...  0.00238575  0.00302755
   0.0016418 ]
 [-0.00379008  0.00362576 -0.00054582 ...  0.00602174 -0.00013645
  -0.01103953]]
[[-2.01425068  0.89171591  0.87148312 ... -0.91281315  0.94890076
  -0.78255797]
 [-2.01425063  0.89171588  0.87148312 ... -0.91281305  0.94890065
  -0.7825579 ]
 [-2.01425064  0.89171593  0.87148317 ... -0.91281306  0.94890073
  -0.78255797]]
f(x_k): -226.77774139478095
